# iTelescope premium image set processing, the easier way


**Author: Dave Strickland**

**Version: 0.5.2-alpha1**

This notebook illustrates how to process a premium image set from iTelescope using the AstroPhotography package. The python environment used corresponds to a miniconda emvironment `ap-env.yml`.

The example dataset used is the iTelescope Plan-20 premium dataset of the M101 supernova [SN 2023 ixf](https://en.wikipedia.org/wiki/SN_2023ixf). Note that these files are not provided with this package. However the notebook should work with your own files if you specify a valid fits-file containing directory at the prompt below.

The processing mirrors that of the `itelescope_premium_the_hard_way.piynb` notebook, except using the `ApProcess` class. Although all of the processing stages allow many user-controlled options the default values will be used in this example.

The processing stages consists of:

1. Creating an `ApProcess` instance to perform the processing. Using the same instance for multiple processing stages on the same set of files is recommended, and simplfies identifying which images to process.
2. (To be added later) Optional bad pixel, bad column and bad row removal in the cases where the input images still have such artifacts. This is common in the iTelescope Premium image sets, where their calibration does not remove all artifacts.
3. Finding stars in the image to allow later astrometric solution finding and image quality checking.
4. Astrometric solutions using Astrometry.net
5. Resampling and stacking of multiple images onto a common footprint to create a final image in each available band..
6. Three color image composite creation using images from  mutliple bands.

## Notebook environment setup

In [1]:
import os
import pathlib
import sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import time
import math
import subprocess
import shlex

from ccdproc import ImageFileCollection
from astropy.table import Table
from astropy import wcs
import astropy
from astropy.io import fits

import AstroPhotography as ap

SyntaxError: invalid syntax (ApProcess.py, line 27)

In [ ]:
def print_module_version(mod):
    """
    Convenience function to pretty-print Python module mod version.

    Arguments:
        mod {module} -- Imported Python module

    Returns:
        str -- module nam
e and version
    """
    print(f"Using module {mod.__name__:30s}  version: {mod.__version__}")
    return

In [ ]:
# Get Version information
print(f'Python version: {sys.version}')
print_module_version(np)
print_module_version(matplotlib)
print_module_version(astropy)
print_module_version(ap)

In [ ]:
# Enable inline plotting for graphics
# %matplotlib inline
# Set default figure size to be larger
# this may only work in matplotlib 2.0+!
from IPython.core.interactiveshell import InteractiveShell
matplotlib.rcParams['figure.figsize'] = [10.0, 6.0]
# Enable multiple outputs from jupyter cells
InteractiveShell.ast_node_interactivity = "all"

# Pipeline Processing

## Create an ApProcess instance

For convenience we'll also change into the directory containing the files we intend to process.

In [ ]:
default_dir = '/old_lnx/home/dks/Downloads/iTelescopeScratch/Plan-20-M101SN-Pipeline'
print(f'Default directory for input files: {default_dir}')

print('Enter the path to the directory containing the premium image set (or return for the default): ')
wdir = input('Image set path:').strip() or default_dir
try:
    os.chdir(wdir.strip())
    print('Switched directory to ' + os.getcwd())
except:
    print(f'Error, os.chdir threw an exception changing to {wdir}')
    print('Check that the path you supplied is a valid filesystem path.')
    raise

In [ ]:
# Choose the logging level.
loglevel = 'DEBUG'
processor = ap.ApProcess(loglevel)

## Star Detection 

### Basic Parameters

In [ ]:
# At the most basic, only data_dir needs to be specified
data_dir        = r'.'

# If the data_dir contains fits files that we do not want processed, then
# you need to specify either the inclusive and/or exclusive file patterns.
#
# Make sure to check that your pattern works on the command line using `ls`
default_include_pattern = "Calibrated-*-?.fits*"
default_exclude_pattern = None

print(f'Default include file pattern for input files: {default_include_pattern}')
print(f'Default exclude file pattern for input files: {default_exclude_pattern}')

msg = 'Enter new include input file pattern (or return to accept default pattern)'
include_pattern = input(msg).strip() or default_include_pattern
print(f'Using "{include_pattern}" as the input include file pattern.')

msg = 'Enter new exclude input file pattern (or return to accept default pattern)'
exclude_pattern = input(msg).strip() or default_exclude_pattern
print(f'Using "{exclude_pattern}" as the input exclude file pattern.')

### Optional Parameters

In [ ]:
# optional

## Run star finding

Before we run find_stars let us check which files it will use as inputs to find_stars. The processing stage is, unsurprisingly, `'inputs'`.

In [ ]:
# Note we need to specify that the input_suffix is 'fits.gz' not for the inputs, which
# are picked up by the include_pattern, but for the output file types.

process_stage = 'inputs'
input_files, filedir = processor.get_file_names(process_stage, data_dir, include_pattern, exclude_pattern, None, None, '.fits.gz')
for file in input_files:
    print(f'  {file} in directory {filedir}')

We can also see what the other output file names will be. **Note** that when processing it is a convention to place the output files in subdirectories.

In [ ]:
stages = ['srclist', 'regfile', 'plotfile', 'qualfile', 'fwhmplot', 'navimg']
for stage in stages:
    print(f'\nProcessing stage {stage} outputs will be:')
    ofiles, filedir = processor.get_file_names(stage, data_dir, include_pattern, exclude_pattern, None, None, '.fits.gz')
    for file in ofiles:
        print(f'  {file} in directory {filedir}')

This matches what we expect. The order of the files is unimportant at this stage.

Now run find_stars. There are a large number of optional parameters that we will just accept the recommended default values for.

In [ ]:
status = processor.find_stars(data_dir, include_pattern, exclude_pattern, None, None, '.fits.gz')
print(status)

# Versions and Changes

| Version | Date | Description |
|:--------|------|-------------|
| 0.5.2-alpha1 | 2024-04-26 | Created script skeleton. |